In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn import preprocessing
from dgl.data import DGLDataset
import dgl
import time
import networkx as nx
import category_encoders as ce
import torch.nn as nn
import torch.nn.functional as F
import dgl.function as fn
import torch
import tqdm
import math

from typing import *
from sklearn.preprocessing import StandardScaler, Normalizer
import socket
import struct
import random
from sklearn.model_selection import train_test_split
import gc

In [2]:
class Discriminator(nn.Module):
    def __init__(self, n_hidden):
      super(Discriminator, self).__init__()
      self.weight = nn.Parameter(torch.Tensor(n_hidden, n_hidden))
      self.reset_parameters()

    def uniform(self, size, tensor):
      bound = 1.0 / math.sqrt(size)
      if tensor is not None:
        tensor.data.uniform_(-bound, bound)

    def reset_parameters(self):
      size = self.weight.size(0)
      self.uniform(size, self.weight)

    def forward(self, features, summary):
      features = torch.matmul(features, torch.matmul(self.weight, summary))
      return features

In [3]:
p= torch.ones((1, 1000))
n= torch.ones((1, 1000))

In [4]:
class DGI(nn.Module):
    def __init__(self):
      super(DGI, self).__init__()
      # self.encoder = SAGE(ndim_in, ndim_out, edim,  F.relu)
      self.discriminator = Discriminator(1000)
      self.loss = nn.BCEWithLogitsLoss()

    def forward(self, positive, negative):
      # positive = self.encoder(g, n_features, e_features, corrupt=False)
      # negative = self.encoder(g, n_features, e_features, corrupt=True)

      # positive = positive[1]
      # negative = negative[1]

      summary = torch.sigmoid(positive.mean(dim=0))

      positive = self.discriminator(positive, summary)
      negative = self.discriminator(negative, summary)

      l1 = self.loss(positive, torch.ones_like(positive))
      l2 = self.loss(negative, torch.zeros_like(negative))

      return l1 + l2

In [5]:
dgi = DGI()


In [12]:
%%time
epochs = 100

dgi.train()

for epoch in range(epochs):

    loss = dgi(p,n)

    if (epoch+1) % 20 == 0:

        print("Epoch {:05d}  ".format(epoch+1))

Epoch 00020  
Epoch 00040  
Epoch 00060  
Epoch 00080  
Epoch 00100  
CPU times: user 350 ms, sys: 0 ns, total: 350 ms
Wall time: 33.6 ms
